### Criterio de Información de Akaike (AIC) en Modelos de Series de Tiempo

#### 1. ¿Qué es el AIC?

El Criterio de Información de Akaike (AIC), desarrollado por el estadístico Hirotane Akaike en 1974, es una métrica estándar utilizada en econometría y estadística para la selección de modelos.

En el análisis de series de tiempo (como en los modelos de Rezago Distribuido, ARDL o VAR), el AIC nos ayuda a resolver una pregunta fundamental: ¿Cuántos rezagos ($p$) debemos incluir en el modelo?

### 2. El Balance entre Ajuste y Complejidad
El AIC se basa en la teoría de la información y busca encontrar el modelo que contenga la mayor cantidad de información real posible minimizando la pérdida de la misma. Para lograrlo, evalúa dos fuerzas opuestas:

Bondad de Ajuste (Verosimilitud): A medida que agregas más rezagos a un modelo de serie de tiempo, este se ajusta cada vez mejor a los datos históricos y reduce el error de estimación.

Parsimonia (Penalización por complejidad): Si agregas demasiados rezagos, el modelo sobreajusta los datos (overfitting), pierde grados de libertad y su capacidad predictiva futura empeora.

El AIC cuantifica este equilibrio mediante la siguiente ecuacion:$$\text{AIC} = -2 \ln(\text{Verosimilitud del Modelo}) + 2k$$

Donde:

El primer término premiará al modelo por tener un menor error (mejor ajuste).

El segundo término ($2k$, donde $k$ es el número total de parámetros estimados) actúa como una penalización estricta por cada rezago o parámetro adicional que añadas.

### 3. Criterio de Decisión: ¿Cómo se interpreta el valor?

Menor AIC, mejor modelo: A diferencia de una medida de bondad de ajuste común (como el $R^2$), el valor del AIC puede ser negativo, y de hecho, entre menor sea el número, mejor será el modelo.

El rezago óptimo: Cuando usamos diferentes números de rezagos (por ejemplo, desde $1$ hasta $15$), calculamos el AIC para cada uno. El rezago idóneo será aquel que minimice el AIC.

### Empecemos con los calculos

### 1. Importemos las Librerias necesarias

In [8]:
import pandas as pd
from statsmodels.tsa.api import VAR

### 2. Carguemos el archivo csv

In [9]:
#Cargar el archivo CSV
ruta_archivo = r"C:\Users\IIEc\Desktop\diego ss\SeriesTiempo.csv"  # O ajusta tu ruta
df = pd.read_csv(ruta_archivo)

### 3. Preparamos los datos (Dataframe)

In [10]:
# Cambiamos de formato la columna fecha y la volvemos el indice del df
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')
df = df.set_index('fecha').sort_index()

In [11]:
#Borramos la columna "version"
df = df.drop(columns=['version'])

### 4. Hacemos que nuestros datos cumplan los supuestos
Para calcular correctamente el orden de los rezagos y evitar problemas de regresión espuria, utilizamos los datos transformados en primeras diferencias ($\Delta X_t = X_t - X_{t-1}$), asegurando que todas las series sean estacionarias (media y varianza constante).


In [12]:
# Aplicar primera diferencia para cumplir el supuesto de estacionariedad
df_diff = df.diff().dropna()
print (df_diff.head())

            entropia_global  entropia_conectados  inflacion_quincenal_deses
fecha                                                                      
2013-01-15        -0.007311            -0.022733                  -0.109428
2013-01-31         0.001004             0.002198                   0.354805
2013-02-15         0.005262             0.018286                  -0.094113
2013-02-28        -0.017412            -0.019985                  -0.127027
2013-03-15         0.006848             0.010896                   0.320176


### La función `VAR`

La función `VAR` implementada en la librería `statsmodels` es una herramienta econométrica diseñada para estimar **Modelos de Vectores Autorregresivos (VAR)** en Python. Su propósito principal es modelar la interdependencia y la dinámica conjunta de múltiples series de tiempo.

---

### 1. ¿Qué datos necesita?

* **Un DataFrame de Pandas:** Que contenga las series de tiempo en columnas.
* **Frecuencia temporal homogénea:** Las observaciones deben estar espaciadas (en nuestro caso, frecuencia quincenal).
* **Estacionariedad ($I(0)$):** Las series deben cumplir con el supuesto de estacionariedad.

---

### 2. ¿Qué te calcula?

1. **La estimación de los coeficientes matriciales:** Resuelve por Mínimos Cuadrados Ordinarios (OLS) ecuación por ecuación las matrices de coeficientes ($A_j$) que miden el impacto del pasado de cada variable sobre las demás a lo largo de los rezagos definidos.
2. **La matriz de covarianza de los residuos ($\Sigma$):** Cuantifica la dispersión y la correlación simultánea de los errores no explicados del sistema.

In [15]:
#Inicializar el modelo VAR con las variables diferenciadas
model = VAR(df_diff)

c:\Users\IIEc\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)


### La función `.select_order()`

La función `.select_order()` es un método integrado en los modelos VAR, diseñado para automatizar la evaluación y comparación de los criterios de información estadísticos en diferentes órdenes de rezago.

---

### 1. ¿Qué datos necesita?


* **Un objeto VAR :** El cual ya debe haber sido alimentado con el DataFrame y haber cumplido los supuestos.

* **Un parámetro de entrada clave (`maxlags`):** Un número entero que define el límite máximo de rezagos que el algoritmo evaluará de forma secuencial (por ejemplo, `maxlags=15`).



---

### 2. ¿Qué te calcula y qué arroja como resultado?


2. **Criterios de información múltiples:** Para cada orden de rezago probado, calcula de forma simultánea los cuatro principales indicadores de selección: **AIC** (Akaike), **BIC** (Bayesian), **FPE** (Final Prediction Error) y **HQIC** (Hannan-Quinn).


3. **Identificación del óptimo (`.summary()` / `.aic`):** Genera un resumen donde resalta con un asteriscos el valor mínimo alcanzado por cada criterio. Esto te permite extraer directamente el número de rezagos óptimo ($p$) sugerido por el criterio de interés (como el AIC) para utilizarlo en las fases posteriores del análisis multivariado o en tus pruebas de Granger.

### Observacion:
Usamos `maxlag = 15` como un límite de búsqueda razonable para el algoritmo por dos razones prácticas:

1. **Horizonte temporal adecuado:** Como tus datos son quincenales (24 observaciones al año), 15 rezagos equivalen a explorar una memoria de casi 8 meses hacia atrás, un rango ideal para capturar la inercia de las variables sin buscar en el pasado lejano.
2. **Protección de grados de libertad:** Evita que el modelo intente estimar demasiados parámetros innecesarios, lo cual agotaría la muestra de más de 300 observaciones y volvería inestables los cálculos.

In [20]:
# Calcular los criterios de información probando hasta 15 rezagos máximos
lag_selection = model.select_order(maxlags=15)

# Imprimir la tabla resumen completa con AIC, BIC, FPE y HQIC
print(lag_selection.summary())

# Obtener directamente el rezago óptimo sugerido por el AIC
print(
    f'\nEl número de rezagos óptimo según el criterio AIC es: {lag_selection.aic}'
)

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -19.34      -19.31   3.982e-09      -19.33
1       -19.65     -19.50*   2.934e-09      -19.59
2       -19.73      -19.47   2.713e-09      -19.62
3       -19.78      -19.43   2.557e-09     -19.64*
4       -19.76      -19.30   2.614e-09      -19.58
5       -19.75      -19.18   2.636e-09      -19.52
6      -19.80*      -19.12  2.523e-09*      -19.53
7       -19.78      -18.99   2.559e-09      -19.47
8       -19.74      -18.84   2.683e-09      -19.38
9       -19.71      -18.71   2.746e-09      -19.31
10      -19.73      -18.61   2.717e-09      -19.28
11      -19.70      -18.48   2.791e-09      -19.21
12      -19.70      -18.37   2.784e-09      -19.17
13      -19.70      -18.27   2.784e-09      -19.13
14      -19.68      -18.14   2.840e-09      -19.07
15      -19.69      -18.03   2.842e-09      -19.03
-------------------------------